# Prometheus & Grafana monitoring

Checks that Prometheus and Grafana are up, lists active Prometheus targets, and links to the six pre-built dashboards in `Grafana/`.

To use a dashboard: open Grafana at `http://<host>:3000` (default login `admin`/`admin`), go to Dashboards -> New -> Import, and upload the matching JSON file from the repo's `Grafana/` folder:

| File | Covers |
|---|---|
| `00_GPU Health` | DCGM GPU utilization, VRAM, temperature |
| `00_NIM Monitoring` | NIM request rate and latency |
| `01_Data Generation` | Generation throughput and judge scores |
| `02_Data_Curation` | Curator pipeline stage counts |
| `03_Evaluation` | Accuracy/F1 across evaluation runs |
| `04_Finetuning` | Training loss and GPU usage during fine-tuning |


In [ ]:
import os
import requests
from IPython.display import display, HTML

# ==========================================
# Step 1: configuration & endpoints
# ==========================================
print("=== Prometheus & Grafana infrastructure check ===")

PROMETHEUS_URL = "http://prometheus:9090"
GRAFANA_URL = "http://grafana:3000"

# ==========================================
# Step 2: health checks
# ==========================================
def check_service(name, url):
    try:
        response = requests.get(url, timeout=3)
        if response.status_code < 400:
            print(f"OK: {name} is reachable at: {url}")
            return True
        else:
            print(f"WARNING: {name} responded with status code: {response.status_code}")
            return False
    except requests.ConnectionError:
        print(f"FAILED: {name} is not reachable ({url}). Is the service running?")
        return False

print("\n--- Checking running services ---")
prom_status = check_service("Prometheus", PROMETHEUS_URL)
grafana_status = check_service("Grafana", GRAFANA_URL)

# ==========================================
# Step 3: GPU metrics check (DCGM exporter)
# ==========================================
if prom_status:
    print("\n--- Checking Prometheus GPU metrics ---")
    try:
        query_url = f"{PROMETHEUS_URL}/api/v1/query?query=up"
        res = requests.get(query_url, timeout=3).json()
        
        targets = res.get("data", {}).get("result", [])
        print(f"Active Prometheus targets found: {len(targets)}")
        for target in targets:
            job = target.get("metric", {}).get("job", "unknown")
            health = target.get("value", ["", "unknown"])[1]
            print(f"   - Job '{job}': status {health}")
    except Exception as e:
        print(f"WARNING: could not query Prometheus metrics: {e}")

# ==========================================
# Step 4: link to the dashboards
# ==========================================
print("\n--- Dashboard links ---")
display(HTML(f"""
<div style="padding: 15px; border: 1px solid #4CAF50; border-radius: 5px; background-color: #f9f9f9;">
    <h3>Quick links to the monitoring UIs</h3>
    <ul>
        <li><b>Grafana:</b> <a href="{GRAFANA_URL}" target="_blank">{GRAFANA_URL}</a> (dashboards for loss, GPU usage and VRAM)</li>
        <li><b>Prometheus:</b> <a href="{PROMETHEUS_URL}" target="_blank">{PROMETHEUS_URL}</a> (for direct PromQL queries)</li>
    </ul>
    <p><i>If the links don't open, forward the ports over SSH, e.g. <code>-L 3000:localhost:3000 -L 9090:localhost:9090</code>.</i></p>
</div>
"""))